In [ ]:
import gc
from pathlib import Path

import numpy as np
import polars as pl
import polars.selectors as cs

# Configure Polars to print everything
pl.Config.set_tbl_rows(30)  # -1 means display all rows
pl.Config.set_tbl_cols(-1)  # -1 means display all columns


In [ ]:
# I/O Constants
INPUT_BASE_PATH = Path("/kaggle/input/competitions/m5-forecasting-accuracy")
OUTPUT_BASE_PATH = Path("/kaggle/working")

# Feature engineering constants
ID_COLS = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
SALES_LAGS = [1, 2, 3, 4, 7, 14]
ROLLING_SALES_WINDOWS = [2, 3, 4, 7, 14]

## LOAD DATA

In [ ]:
# Calendar data
calendar_schema_overrides = {
    "wm_yr_wk": pl.Int16,
    **{date_type: pl.Int16 for date_type in ["wday", "month", "year"]},
    **{f"snap_{state}": pl.Int16 for state in ["CA", "TX", "WI"]}
}
CALENDAR_DATA = pl.read_csv(
    f"{INPUT_BASE_PATH}/calendar.csv",
    try_parse_dates=True,
    schema_overrides=calendar_schema_overrides,
)

# Sell prices
sell_price_schema_overrides = {
    "wm_yr_wk": pl.Int16,
    "sell_price": pl.Float32,
}
SELL_PRICES = pl.read_csv(
    f"{INPUT_BASE_PATH}/sell_prices.csv",
    schema_overrides=sell_price_schema_overrides
)

# Sales data
SALES_TRAIN_EVALUATION = pl.read_csv(
    INPUT_BASE_PATH / "sales_train_evaluation.csv",
)

In [ ]:
# Merge all raw dataframes

raw_train_df = (
    SALES_TRAIN_EVALUATION
    # Cast keys to categorical on the small wide frame, before unpivoting to ~59M rows
    .with_columns(pl.col(ID_COLS).cast(pl.Categorical))
    .unpivot(
        index=ID_COLS,
        variable_name="d",
        value_name="sales",
    )
    .with_columns(
        pl.col("sales").cast(pl.Int16),
        d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16),
    )
    # Drop string d column, no longer needed.
    # d_index is the key from here on
    .drop("d")
    .join(
        # Initially only joining on a subset of columns
        other=CALENDAR_DATA.select(
            "wm_yr_wk",
            d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16),
        ),
        on="d_index",
        how="left",
    )
    .join(
        other=SELL_PRICES.with_columns(pl.col("store_id", "item_id").cast(pl.Categorical)),
        on=["store_id", "item_id", "wm_yr_wk"],
        how="left",
    )
)

In [ ]:
# Raw inputs are no longer needed, free them from memory
del SALES_TRAIN_EVALUATION, SELL_PRICES
gc.collect()

In [ ]:
# Remove rows where price is null
raw_train_df = raw_train_df.filter(pl.col("sell_price").is_not_null())

# Check no null prices
assert not raw_train_df["sell_price"].is_null().any()

In [ ]:
# Check no duplicates

assert raw_train_df.select(pl.col("id"), pl.col("d_index")).is_duplicated().sum() == 0

In [ ]:
# Check that timeseries are continuous

id_series_lengths = (
    raw_train_df
    .group_by("id")
    .agg(
        d_index_min=pl.col("d_index").min(),
        d_index_max=pl.col("d_index").max(),
    )
    .with_columns(min_max_diff=pl.col("d_index_max") - pl.col("d_index_min"))
    .join(
        other=raw_train_df.group_by("id").len(name="series_length"),
        on="id",
        how="inner",
        validate="1:1"
    )
)

assert id_series_lengths["min_max_diff"].eq(id_series_lengths["series_length"] - 1).all()

## LAGGED SALES FEATURES

In [ ]:
# Add columns directly on main features_df, to avoid expensive joins later on

raw_train_df = raw_train_df.with_columns(
    [
        pl.col("sales")
        .shift(lag)
        .over("id", order_by="d_index")
        .alias(f"sales_lagged_{lag}")
        for lag in SALES_LAGS
    ]
)

raw_train_df.select(["id", "d_index", "sales", *[f"sales_lagged_{lag}" for lag in SALES_LAGS]]).head()

## ROLLING SALES FEATURES

In [ ]:
# Add columns directly on main features_df, to avoid expensive joins later on

raw_train_df = raw_train_df.with_columns(
    [
        pl.col("sales")
        .shift(1)
        .rolling_mean(window_size=window, min_samples=1)
        .over("id", order_by="d_index")
        .alias(f"sales_rolling_mean_{window}")
        .cast(pl.Float32)
        for window in ROLLING_SALES_WINDOWS
    ]
)

raw_train_df.select(["id", "d_index", "sales", *[f"sales_rolling_mean_{w}" for w in ROLLING_SALES_WINDOWS]]).head()

## LAGGED PRICE FEATURES

### TODO

## ROLLING PRICE FEATURES

### TODO

## EVENT FEATURES

In [ ]:
N_EVENTS = 2

EVENT_FEATURES_DF = (
    CALENDAR_DATA
    .select(["wm_yr_wk", "d", "date", "event_name_1", "event_type_1", "event_name_2", "event_type_2"])
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16))
    .with_columns(
        pl.when(pl.col(f"event_name_{i}").is_not_null())
        .then(pl.col("d_index"))
        .otherwise(None)
        .alias(f"event_index_{i}")
        for i in (1, 2)
    )
    # Days until next / since last event
    .with_columns(
        # Days since last event
        *[
            (pl.col("d_index") - pl.col(f"event_index_{i}").forward_fill())
            .alias(f"days_since_event_{i}")
            for i in (1, 2)
        ],
        # Last event type
        *[
            (pl.col(f"event_type_{i}").forward_fill()
            .alias(f"last_event_type_{i}"))
            for i in range(1, N_EVENTS + 1)
        ],
        # Days until next event
        *[
            (pl.col(f"event_index_{i}").backward_fill() - pl.col("d_index"))
            .alias(f"days_until_event_{i}")
            for i in (1, 2)
        ],
        # Next event type
        *[
            (pl.col(f"event_type_{i}").backward_fill()
            .alias(f"next_event_type_{i}"))
            for i in range(1, N_EVENTS + 1)
        ],
    )
    .with_columns(
        # Clip days since last event and event type to 7 days.
        *[
            pl.when(pl.col(f"days_since_event_{i}") <= 7)
            .then(pl.col(f"days_since_event_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            pl.when(pl.col(f"days_since_event_{i}") <= 7).
            then(pl.col(f"last_event_type_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
        # Clip days until next event and event type to 7 days.
        *[              
            pl.when(pl.col(f"days_until_event_{i}") <= 7)
            .then(pl.col(f"days_until_event_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            pl.when(pl.col(f"days_until_event_{i}") <= 7)
            .then(pl.col(f"next_event_type_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ]
    )
)

EVENT_FEATURES_DF.head()

## CALENDAR FEATURS

In [ ]:
CALENDAR_FEATURES_DF = (
    CALENDAR_DATA
    .select(["wm_yr_wk", "d", "date", "wday", "month", "year"])
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16))
    .with_columns(
        sin_wday=(2 * np.pi * pl.col("wday") / 7).sin().cast(pl.Float32),
        cos_wday=(2 * np.pi * pl.col("wday") / 7).cos().cast(pl.Float32),
        sin_month=(2 * np.pi * pl.col("month") / 12).sin().cast(pl.Float32),
        cos_month=(2 * np.pi * pl.col("month") / 12).cos().cast(pl.Float32),
        is_weekend=(pl.col("wday") >= 6).cast(pl.Float32),
    )
)

CALENDAR_FEATURES_DF.head()

## MERGE ALL FEATURES

In [ ]:
# Small (1969 rows) calendar-level frame: drop string/date keys and make remaining strings categorical
# before it is broadcast onto the full train frame.
CALENDAR_ALL_FEATURES_DF = (
    CALENDAR_FEATURES_DF.drop("d", "date")
    .join(
        other=EVENT_FEATURES_DF.drop("d", "date"),
        on=["wm_yr_wk", "d_index"],
        how="inner",
        validate="1:1",
    )
    .with_columns(cs.string().cast(pl.Categorical))
)

# Lag and rolling features were already added to raw_train_df in place, so only the calendar join remains
FULL_TRAIN_DF = raw_train_df.join(
    other=CALENDAR_ALL_FEATURES_DF,
    on=["wm_yr_wk", "d_index"],
    how="inner",
    validate="m:1",
)

del raw_train_df
gc.collect()

In [ ]:
# Write to output
FULL_TRAIN_DF.write_parquet(OUTPUT_BASE_PATH / "m5-acc-features-df.pq")